In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field

In [2]:
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [3]:
from src.gepa.scaffold import GepaConfig, run_optimization_pipeline
from src.gepa.data_utils import create_dataset_from_dicts
from src.gepa.lm import get_openai_model
from src.gepa.types import DataInstWithInput, RolloutOutput

In [4]:
class PassengerInput(BaseModel):
    """Input features for Titanic survival prediction."""
    
    passenger_class: Literal[1, 2, 3] = Field(
        description="Passenger class (1=First, 2=Second, 3=Third)"
    )
    sex: Literal["male", "female"] = Field(
        description="Passenger's sex"
    )
    age: float = Field(
        description="Passenger's age in years"
    )
    siblings_spouses: int = Field(
        description="Number of siblings/spouses aboard"
    )
    parents_children: int = Field(
        description="Number of parents/children aboard"
    )
    fare: float = Field(
        description="Passenger fare paid in pounds"
    )
    embarked: Literal["C", "Q", "S", "Unknown"] = Field(
        description="Port of embarkation (C=Cherbourg, Q=Queenstown, S=Southampton)"
    )


class SurvivalPrediction(BaseModel):
    """Output prediction for Titanic survival."""
    
    survived: Literal["yes", "no"] = Field(
        description="Whether the passenger survived"
    )
    confidence: float = Field(
        description="Confidence score between 0 and 1",
        ge=0.0,
        le=1.0
    )
    reasoning: str = Field(
        description="Detailed explanation of the prediction based on the features"
    )

In [8]:
import pandas as pd


def load_titanic_data(n_train: int = 50, n_holdout: int = 15) -> tuple[list[dict], list[dict]]:
    """Load and prepare Titanic dataset for GEPA with holdout test set.
    
    Args:
        n_train: Number of samples to use for training/validation (default 50)
        n_holdout: Number of samples to hold out for final testing (default 15)
        
    Returns:
        Tuple of (training_data, holdout_data) as lists of dictionaries
    """
    # Load Titanic dataset from seaborn
    try:
        import seaborn as sns
        df = sns.load_dataset('titanic')
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Make sure seaborn is installed: pip install seaborn")
        raise
    
    # Select interesting features and clean data
    df = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']].copy()
    
    # Drop rows with missing critical values
    df = df.dropna(subset=['age', 'fare'])
    
    # Fill missing embarked with 'Unknown'
    df['embarked'] = df['embarked'].fillna('Unknown')
    
    # Convert survived to string labels
    df['survived_label'] = df['survived'].map({0: 'no', 1: 'yes'})
    
    # Sample diverse examples - stratified by survival and class
    # First, separate into train and holdout sets
    train_dfs = []
    holdout_dfs = []
    
    for survived in [0, 1]:
        for pclass in [1, 2, 3]:
            subset = df[(df['survived'] == survived) & (df['pclass'] == pclass)]
            if len(subset) > 0:
                # Calculate proportional samples for this stratum
                n_train_stratum = min(len(subset), max(1, n_train // 6))
                n_holdout_stratum = min(len(subset) - n_train_stratum, max(1, n_holdout // 6))
                
                # Shuffle and split
                subset_shuffled = subset.sample(frac=1.0, random_state=42)
                train_dfs.append(subset_shuffled.iloc[:n_train_stratum])
                
                if n_holdout_stratum > 0 and len(subset_shuffled) > n_train_stratum:
                    holdout_dfs.append(subset_shuffled.iloc[n_train_stratum:n_train_stratum + n_holdout_stratum])
    
    # Combine and limit to requested sizes
    df_train = pd.concat(train_dfs, ignore_index=True)
    if len(df_train) > n_train:
        df_train = df_train.sample(n=n_train, random_state=42)
    
    df_holdout = pd.concat(holdout_dfs, ignore_index=True) if holdout_dfs else pd.DataFrame()
    if len(df_holdout) > n_holdout:
        df_holdout = df_holdout.sample(n=n_holdout, random_state=43)
    
    # Convert to list of dicts
    def df_to_dict_list(df: pd.DataFrame) -> list[dict]:
        """Convert DataFrame to list of dictionaries."""
        data = []
        for _, row in df.iterrows():
            data.append({
                'passenger_class': int(row['pclass']),
                'sex': str(row['sex']),
                'age': float(row['age']),
                'siblings_spouses': int(row['sibsp']),
                'parents_children': int(row['parch']),
                'fare': float(row['fare']),
                'embarked': str(row['embarked']),
                'label': str(row['survived_label'])
            })
        return data
    
    train_data = df_to_dict_list(df_train)
    holdout_data = df_to_dict_list(df_holdout)
    
    return train_data, holdout_data

In [9]:
def survival_metric(
    data_inst: DataInstWithInput[PassengerInput],
    output: RolloutOutput[SurvivalPrediction],
) -> tuple[float, str | None]:
    """Evaluate survival prediction accuracy.
    
    This metric checks if the predicted survival matches the ground truth.
    It also considers confidence calibration as a bonus.
    
    Args:
        data_inst: Input data instance with metadata containing ground truth.
        output: Agent's output to evaluate.
        
    Returns:
        Tuple of (score, feedback) where score is between 0.0 and 1.0.
    """
    # Check if the agent execution was successful
    if not output.success or output.result is None:
        return 0.0, output.error_message or "Agent failed to produce output"
    
    # Extract predicted survival
    predicted_survival = output.result.survived
    confidence = output.result.confidence
    
    # Extract ground truth from metadata
    ground_truth = data_inst.metadata.get("label")
    
    if ground_truth is None:
        return 0.0, "No ground truth label found in metadata"
    
    # Base score: correct prediction gets 1.0, incorrect gets 0.0
    if predicted_survival == ground_truth:
        # Bonus for high confidence on correct predictions
        score = 0.7 + (0.3 * confidence)
        feedback = f"✓ Correct: {predicted_survival} (confidence: {confidence:.2f})"
    else:
        # Penalty scales with confidence on wrong predictions
        score = 0.3 * (1 - confidence)
        feedback = f"✗ Incorrect: predicted {predicted_survival}, expected {ground_truth} (confidence: {confidence:.2f})"
    
    return score, feedback

In [50]:
train_data, holdout_data = load_titanic_data(n_train=50, n_holdout=50)

print(f"Loaded {len(train_data)} training records")
print(f"Loaded {len(holdout_data)} holdout test records")

Loaded 48 training records
Loaded 48 holdout test records


In [46]:
train_survival_counts = {}
for record in train_data:
    label = record['label']
    train_survival_counts[label] = train_survival_counts.get(label, 0) + 1

holdout_survival_counts = {}
for record in holdout_data:
    label = record['label']
    holdout_survival_counts[label] = holdout_survival_counts.get(label, 0) + 1

print(f"Training survival distribution: {train_survival_counts}")
print(f"Holdout survival distribution: {holdout_survival_counts}")

Training survival distribution: {'no': 60, 'yes': 60}
Holdout survival distribution: {'no': 24, 'yes': 24}


In [51]:
dataset = create_dataset_from_dicts(
    train_data,
    input_model=PassengerInput,
    input_keys=['passenger_class', 'sex', 'age', 'siblings_spouses', 
                'parents_children', 'fare', 'embarked'],
    metadata_keys=['label'],
)

print(f"Created training dataset with {len(dataset)} examples")

Created training dataset with 48 examples


In [ ]:
reflection_model = "gpt-4.1"
agent_model="gpt-4.1-mini"


config = GepaConfig(
    # Agent configuration
    agent_model=agent_model,
    agent_instructions=(
        "Predict Titanic survival based on the passenger's features."
    ),
    input_type=PassengerInput,
    output_type=SurvivalPrediction,
    
    # Data and evaluation
    dataset=dataset,
    metric=survival_metric,
    train_ratio=0.60,
    
    # Optimization parameters
    max_metric_calls=200,  # More calls for better optimization
    module_selector="round_robin",  # Optimize instructions, signature, and tools
    candidate_selection_strategy="pareto",
    optimize_tools=True,
    use_merge=True,
    
    # LLM for reflection
    reflection_model=reflection_model,
    
    # Display options
    display_progress_bar=True,
    track_best_outputs=True,
    
    # Caching for faster iterations
    enable_cache=True,
    cache_dir=".gepa_cache",
    
    # Output settings
    output_dir="optimization_results",
    save_result=True,
)

In [63]:
import nest_asyncio

nest_asyncio.apply()


result = run_optimization_pipeline(config)

Dataset split: 36 training, 12 validation examples
Starting GEPA optimization...


GEPA Optimization:   0%|          | 0/200 [00:00<?, ?rollouts/s]

Iteration 0: Base program full valset score: 0.0 over 12 / 12 examples
Iteration 1: Selected program 0 score: 0.0
Iteration 1: Proposed new text for instructions: Given the provided passenger features, predict whether the passenger survived the Titanic disaster. Use known patterns from Titanic data for prediction: 

- Female passengers had a much higher chance of survival, especially in 1st and 2nd class.
- 1st class passengers had a higher survival rate than 2nd or 3rd class.
- Children (age <16) were prioritized for lifeboats, especially if traveling with family.
- Higher fare is strongly correlated with higher chance of survival.
- Passengers embarking from Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q).
- Traveling with family (siblings/spouses, parents/children >0) may slightly increase survival odds for some demographics (notably children).

For each prediction, provide:
- Whether the passenger survived (yes/no)
- A confidence score betw

GEPA Optimization:  15%|█▌        | 30/200 [00:28<02:41,  1.05rollouts/s]

Iteration 1: Found a better program on the valset with score 0.73375.
Iteration 1: Valset score for new program: 0.73375 (coverage 12 / 12)
Iteration 1: Val aggregate for new program: 0.73375
Iteration 1: Individual valset scores for new program: {0: 0.075, 1: 0.955, 2: 0.955, 3: 0.075, 4: 0.955, 5: 0.955, 6: 0.94, 7: 0.955, 8: 0.955, 9: 0.955, 10: 0.075, 11: 0.955}
Iteration 1: New valset pareto front scores: {0: 0.075, 1: 0.955, 2: 0.955, 3: 0.075, 4: 0.955, 5: 0.955, 6: 0.94, 7: 0.955, 8: 0.955, 9: 0.955, 10: 0.075, 11: 0.955}
Iteration 1: Valset pareto front aggregate score: 0.73375
Iteration 1: Updated valset pareto front programs: {0: {1}, 1: {1}, 2: {1}, 3: {1}, 4: {1}, 5: {1}, 6: {1}, 7: {1}, 8: {1}, 9: {1}, 10: {1}, 11: {1}}
Iteration 1: Best valset aggregate score so far: 0.73375
Iteration 1: Best program as per aggregate score on valset: 1
Iteration 1: Best score on valset: 0.73375
Iteration 1: Linear pareto front program index: 1
Iteration 1: New program candidate index: 1


...

Iteration 13: New subsample score 2.925 is not better than old score 2.925, skipping
Iteration 14: Selected program 7 score: 0.73625
Iteration 14: Proposed new text for signature:PassengerInput:siblings_spouses:desc: Number of siblings or spouses aboard. This feature partially reflects family size and can influence survival. For children (age < 16), having siblings or a parent aboard often increased survival odds as children in families were prioritized during evacuation. For adults, especially in 3rd class, traveling with a large family sometimes reduced survival chances due to crowding, slower evacuation, and less access to lifeboats. For 1st and 2nd class, family presence had less negative impact and could even marginally improve survival by facilitating group rescue. Always consider this feature in combination with age, passenger class, and sex for accurate survival prediction.
Iteration 14: New subsample score 1.9699999999999998 is better than old score 1.955. Continue to full eva

GEPA Optimization:  96%|█████████▌| 191/200 [06:26<00:19,  2.13s/rollouts]

Iteration 14: Found a better program on the valset with score 0.7374999999999999.
Iteration 14: Valset score for new program: 0.7374999999999999 (coverage 12 / 12)
Iteration 14: Val aggregate for new program: 0.7374999999999999
Iteration 14: Individual valset scores for new program: {0: 0.09000000000000001, 1: 0.97, 2: 0.97, 3: 0.09000000000000001, 4: 0.955, 5: 0.955, 6: 0.955, 7: 0.955, 8: 0.955, 9: 0.955, 10: 0.045000000000000005, 11: 0.955}
Iteration 14: New valset pareto front scores: {0: 0.09000000000000001, 1: 0.97, 2: 0.97, 3: 0.09000000000000001, 4: 0.97, 5: 0.97, 6: 0.955, 7: 0.955, 8: 0.955, 9: 0.97, 10: 0.09000000000000001, 11: 0.955}
Iteration 14: Valset pareto front aggregate score: 0.745
Iteration 14: Updated valset pareto front programs: {0: {8, 2, 3}, 1: {8}, 2: {8, 5, 7}, 3: {2, 3, 5, 6, 7, 8}, 4: {5}, 5: {7}, 6: {8, 7}, 7: {1, 2, 3, 4, 5, 7, 8}, 8: {1, 2, 3, 5, 6, 7, 8}, 9: {5}, 10: {2}, 11: {1, 2, 3, 5, 6, 7, 8}}
Iteration 14: Best valset aggregate score so far: 0.73

GEPA Optimization:  96%|█████████▌| 191/200 [06:51<00:19,  2.16s/rollouts]

Iteration 15: Valset score for new program: 0.735 (coverage 12 / 12)
Iteration 15: Val aggregate for new program: 0.735
Iteration 15: Individual valset scores for new program: {0: 0.09000000000000001, 1: 0.955, 2: 0.97, 3: 0.09000000000000001, 4: 0.955, 5: 0.955, 6: 0.955, 7: 0.94, 8: 0.955, 9: 0.955, 10: 0.045000000000000005, 11: 0.955}
Iteration 15: New valset pareto front scores: {0: 0.09000000000000001, 1: 0.97, 2: 0.97, 3: 0.09000000000000001, 4: 0.97, 5: 0.97, 6: 0.955, 7: 0.955, 8: 0.955, 9: 0.97, 10: 0.09000000000000001, 11: 0.955}
Iteration 15: Valset pareto front aggregate score: 0.745
Iteration 15: Updated valset pareto front programs: {0: {8, 9, 2, 3}, 1: {8}, 2: {8, 9, 5, 7}, 3: {2, 3, 5, 6, 7, 8, 9}, 4: {5}, 5: {7}, 6: {8, 9, 7}, 7: {1, 2, 3, 4, 5, 7, 8}, 8: {1, 2, 3, 5, 6, 7, 8, 9}, 9: {5}, 10: {2}, 11: {1, 2, 3, 5, 6, 7, 8, 9}}
Iteration 15: Best valset aggregate score so far: 0.7374999999999999
Iteration 15: Best program as per aggregate score on valset: 8
Iteration 15

In [64]:
print(f"Best Score: {result.best_score:.4f}")

if result.original_score is not None:
    print(f"Original Score: {result.original_score:.4f}")
    improvement = result.improvement_ratio()
    if improvement is not None:
        print(f"Improvement: {improvement:+.2%}")

print(f"Iterations: {result.num_iterations}")
print(f"Metric Calls: {result.num_metric_calls}")
print(f"GEPA Input Tokens: {result.gepa_usage.input_tokens}")
print(f"GEPA Output Tokens: {result.gepa_usage.output_tokens}")


print("\nOptimized Components:")
for component_name, component_value in result.best_candidate.items():
    print(f"\n{component_name}:")
    print(f"  {component_value}")

Best Score: 0.7375
Original Score: 0.0000
Iterations: 10
Metric Calls: 208
GEPA Input Tokens: 151634
GEPA Output Tokens: 25778

Optimized Components:

instructions:
  Given the provided passenger features, predict whether the passenger survived the Titanic disaster. Use known patterns from Titanic data for prediction: 

- Female passengers had a much higher chance of survival, especially in 1st and 2nd class.
- 1st class passengers had a higher survival rate than 2nd or 3rd class.
- Children (age <16) were prioritized for lifeboats, especially if traveling with family.
- Higher fare is strongly correlated with higher chance of survival.
- Passengers embarking from Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q).
- Traveling with family (siblings/spouses, parents/children >0) may slightly increase survival odds for some demographics (notably children).

For each prediction, provide:
- Whether the passenger survived (yes/no)
- A confidence score 

In [65]:
result.best_candidate

{'instructions': 'Given the provided passenger features, predict whether the passenger survived the Titanic disaster. Use known patterns from Titanic data for prediction: \n\n- Female passengers had a much higher chance of survival, especially in 1st and 2nd class.\n- 1st class passengers had a higher survival rate than 2nd or 3rd class.\n- Children (age <16) were prioritized for lifeboats, especially if traveling with family.\n- Higher fare is strongly correlated with higher chance of survival.\n- Passengers embarking from Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q).\n- Traveling with family (siblings/spouses, parents/children >0) may slightly increase survival odds for some demographics (notably children).\n\nFor each prediction, provide:\n- Whether the passenger survived (yes/no)\n- A confidence score between 0 and 1\n- Detailed reasoning citing relevant features (e.g., class, sex, age, fare, family onboard, port of embarkation)\n\nExamp

In [66]:
from IPython.display import Markdown

Markdown(result.best_candidate['instructions'])

Given the provided passenger features, predict whether the passenger survived the Titanic disaster. Use known patterns from Titanic data for prediction: 

- Female passengers had a much higher chance of survival, especially in 1st and 2nd class.
- 1st class passengers had a higher survival rate than 2nd or 3rd class.
- Children (age <16) were prioritized for lifeboats, especially if traveling with family.
- Higher fare is strongly correlated with higher chance of survival.
- Passengers embarking from Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q).
- Traveling with family (siblings/spouses, parents/children >0) may slightly increase survival odds for some demographics (notably children).

For each prediction, provide:
- Whether the passenger survived (yes/no)
- A confidence score between 0 and 1
- Detailed reasoning citing relevant features (e.g., class, sex, age, fare, family onboard, port of embarkation)

Example:
Input: 1st class, female, 24 years old, 0 siblings/spouses, 0 parents/children, £100 fare, embarked Cherbourg
Output: Survived: Yes
Confidence: 0.95
Reasoning: Female and 1st class passengers had much higher survival rates. High fare further increases the chance. Cherbourg embarkation is also associated with higher survival rates.

In [67]:
Markdown(result.best_candidate['tool:final_result:description'])

Given the provided passenger features, predict whether the passenger survived the Titanic disaster. Your prediction should reflect detailed domain-specific patterns found in Titanic data, including:

- Female passengers had a much higher chance of survival, especially in 1st and 2nd class.
- 1st class passengers had a notably higher survival rate than 2nd or 3rd class. 3rd class males, in particular, rarely survived.
- Young children (age < 16), especially infants, received priority for lifeboats. Traveling with family (siblings/spouses or parents/children >0) further increased survival odds for children.
- Higher fares typically indicated higher social status and increased chance of survival.
- Passengers embarking from Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q).
- Large family groups sometimes reduced adult survival odds in 3rd class but increased for children.
- Survival rates for 3rd class males, regardless of age, fare, or family, were especially low compared to other groups.
- Context: Some rare exceptions exist—children and females in 3rd class with high fares and large families still often perished.

For each prediction, provide:
- Whether the passenger survived (yes/no)
- A confidence score between 0 (no chance) and 1 (certain)
- A detailed, step-by-step reasoning. For clarity, specifically reference each relevant input feature—class, sex, age, fare, family onboard (both siblings/spouses and parents/children), and port of embarkation—in making your determination, explicitly noting when a factor is especially influential (either positively or negatively).

Example:
Input: 3rd class, male, 1 year old, 5 siblings/spouses, 2 parents/children, £46.9 fare, embarked Southampton
Output: Survived: No
Confidence: 0.05
Reasoning: Although the passenger is a male infant (which increases survival odds), he is in 3rd class with a large family and embarked from Southampton. Despite a relatively high fare, almost all 3rd class males (even young children) did not survive, especially in large families. The negative impact of class, sex, and family size in this context outweighs the positive effects of age and fare, resulting in a very low chance of survival.

In [68]:
Markdown(result.best_candidate['signature:PassengerInput:sex:desc'])

Passenger's sex. Strongly influences survival: Females generally had a much higher chance of survival across all classes, especially in 1st and 2nd. Males had significantly lower survival rates, particularly in 3rd class, with the exception of young boys (under 16) who had a slightly increased chance compared to adult males. Always explicitly consider the impact of sex on survival when reasoning.

In [69]:
Markdown(result.best_candidate['signature:PassengerInput:siblings_spouses:desc'])

Number of siblings or spouses aboard. This feature partially reflects family size and can influence survival. For children (age < 16), having siblings or a parent aboard often increased survival odds as children in families were prioritized during evacuation. For adults, especially in 3rd class, traveling with a large family sometimes reduced survival chances due to crowding, slower evacuation, and less access to lifeboats. For 1st and 2nd class, family presence had less negative impact and could even marginally improve survival by facilitating group rescue. Always consider this feature in combination with age, passenger class, and sex for accurate survival prediction.

In [70]:
from gepa.gepa_utils import find_dominator_programs

pareto_front_programs = find_dominator_programs(result.raw_result.per_val_instance_best_candidates, result.raw_result.val_aggregate_scores)

In [71]:
def dag_to_dot(parent_program_for_candidate, dominator_program_ids, best_program_idx, full_eval_scores):
    dot_lines = [
        "digraph G {",
        "    node [style=filled, shape=circle, fontsize=50];"
    ]
    n = len(parent_program_for_candidate)
    # Set up nodes with colors and scores in labels
    for idx in range(n):
        score = full_eval_scores[idx]
        label = f"{idx}\\n({score:.2f})"
        if idx == best_program_idx:
            dot_lines.append(f'    {idx} [label="{label}", fillcolor=cyan, fontcolor=black];')
        elif idx in dominator_program_ids:
            dot_lines.append(f'    {idx} [label="{label}", fillcolor=orange, fontcolor=black];')
        else:
            dot_lines.append(f'    {idx} [label="{label}"];')
    
    # Set up edges
    for child, parents in enumerate(parent_program_for_candidate):
        for parent in parents:
            if parent is not None:
                dot_lines.append(f'    {parent} -> {child};')
    
    dot_lines.append("}")
    return "\n".join(dot_lines)

In [72]:
print(dag_to_dot(
    result.raw_result.parents,
    pareto_front_programs,
    result.raw_result.best_idx,
    result.raw_result.val_aggregate_scores
))

digraph G {
    node [style=filled, shape=circle, fontsize=50];
    0 [label="0\n(0.00)"];
    1 [label="1\n(0.73)"];
    2 [label="2\n(0.74)", fillcolor=orange, fontcolor=black];
    3 [label="3\n(0.73)"];
    4 [label="4\n(0.72)"];
    5 [label="5\n(0.73)", fillcolor=orange, fontcolor=black];
    6 [label="6\n(0.73)"];
    7 [label="7\n(0.74)", fillcolor=orange, fontcolor=black];
    8 [label="8\n(0.74)", fillcolor=cyan, fontcolor=black];
    9 [label="9\n(0.73)"];
    0 -> 1;
    1 -> 2;
    2 -> 3;
    3 -> 4;
    3 -> 5;
    2 -> 6;
    2 -> 7;
    7 -> 8;
    5 -> 9;
    8 -> 9;
}


In [75]:
# from IPython.display import display
# from graphviz import Source


# dot_string = dag_to_dot(
#     result.raw_result.parents,
#     pareto_front_programs,
#     result.raw_result.best_idx,
#     result.raw_result.val_aggregate_scores
# )

# display(Source(dot_string))

In [ ]:
data\plots\graphviz.svg